<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [11]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [12]:
from bootcamp_agent.checks import check, review

## 1. Warm-up: weak prompt vs project-aware prompt

In your assistant, ask both and paste short excerpts of the answers here:

- Weak: *"add a search feature"*
- Project-aware: *"read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

**Paste both responses in the cell below and note the differences.**

**add a search feature**

The repository already has lexical search internally via `search_documents`, but it is not exposed as a direct user-facing command.

Which interface should I add?

1. CLI: `uv run bootcamp-agent --search "prompt injection"` with ranked passages
2. Agent behavior: improve natural-language question search
3. Notebook/workspace: add a search cell or UI
4. Something else

My default recommendation is option 1: expose the existing bounded retrieval through the CLI without adding dependencies or changing the retrieval algorithm.

**read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits**

**Proposed Plan**

1. Update `tools.py:37-50` so `search_documents` accepts an optional `tag: str | None = None`.
2. Validate the tag at the tool boundary:
   - Reject empty or unknown tags with `ToolError`.
   - Include the valid tag names in the error message.
3. Filter documents by exact membership in `doc.tags` before calling `retrieve`.
4. Preserve existing query validation, result clamping, output format, and no-results behavior.
5. Update the tool description to document the optional tag filter.
6. Validate with a focused runtime check covering:
   - unfiltered search,
   - matching tag,
   - valid tag with no query matches,
   - empty tag,
   - unknown tag.
7. Run `uv run ruff check src/bootcamp_agent/tools.py`.

No files will be edited until you approve the plan.


## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason:** *The assistant suggested refactoring silent clamping of max_results into explicit rejection (point 2) and adding case-insensitive tag matching (point 3). Rejected because both are out of scope — the task is only to add a tags filter, and changing existing validation patterns could break tests and violates AGENTS.md's minimal-change principle.*

## 3. Improve the instructions

Where did the assistant assume wrong? That sentence belongs in `AGENTS.md`. Make the edit, note it here — instructions are code (see `docs/guides/harness-engineering.md`).

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [13]:
loop = {
    "plan_approved": "It proposed editing only tools.py: add an optional tag parameter to search_documents, filter results after retrieve, raise ToolError for empty or unknown tags. No new dependencies.",
    "diff_inspected": "Added a tag keyword argument defaulting to None in search_documents. If tag is provided, it validates against known tags from the corpus and filters matching documents before scoring. Empty string and unknown tags raise ToolError listing valid options.",
    "rejected_change": "Points 2 and 3 from its improvement suggestions: refactoring silent clamping of max_results into explicit rejection, and adding case-insensitive tag matching.",
    "why_rejected": "Out of scope. The task is only to add a tags filter. Changing existing validation patterns could break tests and goes beyond what the lesson asks for, violating AGENTS.md minimal-change principle.",
    "risks": "No regression coverage since tests/ is withheld. Unknown grading contract could break if it introspects the old 2-param signature. Ambiguous empty-result message does not distinguish zero matches from tag filtering out everything.",
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")

plan_approved      It proposed editing only tools.py: add an optional tag par
diff_inspected     Added a tag keyword argument defaulting to None in search_
rejected_change    Points 2 and 3 from its improvement suggestions: refactori
why_rejected       Out of scope. The task is only to add a tags filter. Chang
risks              No regression coverage since tests/ is withheld. Unknown g


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [14]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [15]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.